# 🌿 Crop Disease Classification
### EfficientNetB3 + Custom CNN Head | PlantVillage + PlantDoc

**Pipeline:**
1. Mount Google Drive & install dependencies
2. Load & preprocess datasets
3. Build model (EfficientNetB3 backbone + custom head)
4. Phase 1 — Train head only
5. Phase 2 — Fine-tune top backbone layers
6. Evaluate & save model

## Step 0 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Update this path to where your processed/ folder is in Drive ──
#DRIVE_BASE = '/content/drive/MyDrive/processed'

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1 — Imports & Constants

In [ ]:
import zipfile
from pathlib import Path

zip_path   = '/content/drive/MyDrive/processed.zip'
extract_to = '/content/'

print("Unzipping to Colab local storage...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Done!")



Unzipping to Colab local storage...
Done!


In [ ]:
# Then update your paths
TRAIN_DIR = Path('/content/processed/train')
VAL_DIR   = Path('/content/processed/val')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {tf.config.list_physical_devices("GPU")}')

TensorFlow version : 2.20.0
GPU available      : []


In [ ]:
# ── Paths ────────────────────────────────────────────────
#TRAIN_DIR      = Path(DRIVE_BASE) / 'train'
#VAL_DIR        = Path(DRIVE_BASE) / 'val'
MODEL_SAVE_DIR = Path('/content/drive/MyDrive/PlantDisease/model/saved')
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────
IMG_SIZE    = (300, 300)   # EfficientNetB3 native input size
BATCH_SIZE  = 32
SEED        = 42
NUM_CLASSES = 38

print('Paths configured.')
print(f'Train dir exists : {TRAIN_DIR.exists()}')
print(f'Val dir exists   : {VAL_DIR.exists()}')

Paths configured.
Train dir exists : True
Val dir exists   : True


## Step 2 — Data Loaders

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=SEED,
    label_mode='categorical'
)

class_names = train_ds.class_names
print(f'Classes found  : {len(class_names)}')
print(f'Train batches  : {len(train_ds)}')
print(f'Val batches    : {len(val_ds)}')

Found 72963 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.
Classes found  : 38
Train batches  : 2281
Val batches    : 550


## Step 3 — Preprocessing & Augmentation

In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping Colab alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

<IPython.core.display.Javascript object>

In [ ]:
preprocess = tf.keras.applications.efficientnet.preprocess_input

augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2),
], name='augmentation')

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=SEED,
    label_mode='categorical'
)

train_ds = (train_ds
    .map(lambda x, y: (preprocess(augmentation(x, training=True)), y),
         num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (val_ds
    .map(lambda x, y: (preprocess(x), y),
         num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Pipeline ready — no cache, prefetch only.")

Found 72963 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.
Pipeline ready — no cache, prefetch only.


## Step 4 — Model Architecture

In [ ]:
# ── Backbone ──────────────────────────────────────────────
backbone = EfficientNetB3(
    include_top=False,
    weights='imagenet',
    input_shape=(300, 300, 3)
)
backbone.trainable = False   # freeze all backbone layers

# ── Custom Classification Head ────────────────────────────
inputs  = tf.keras.Input(shape=(300, 300, 3))
x       = backbone(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs, name='CropDiseaseClassifier')
model.summary()

trainable     = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable = sum([tf.size(w).numpy() for w in model.non_trainable_weights])
print(f'\nTrainable parameters     : {trainable:,}')
print(f'Non-trainable parameters : {non_trainable:,}')

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "CropDiseaseClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 10, 10, 1536)   │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,192,917 (42.70 MB)

 Trainable params: 406,310 (1.55 MB)

 Non-trainable params: 10,786,607 (41.15 MB)


Trainable parameters     : 406,310
Non-trainable parameters : 10,786,607


## Step 5 — Compile & Callbacks

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_SAVE_DIR / 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print('Model compiled.')
print('Callbacks ready.')

Model compiled.
Callbacks ready.


## Step 6 — Phase 1: Train Head Only

In [ ]:
print('=' * 50)
print('PHASE 1 — Training classification head')
print('=' * 50)

EPOCHS_PHASE1 = 20

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks
)

print('\nPhase 1 complete.')
print(f'Best val_accuracy : {max(history_phase1.history["val_accuracy"]):.4f}')

PHASE 1 — Training classification head
Epoch 1/20
2281/2281 ━━━━━━━━━━━━━━━━━━━━ 0s 654ms/step - accuracy: 0.7391 - loss: 0.9273
Epoch 1: val_accuracy improved from None to 0.94429, saving model to /content/drive/MyDrive/PlantDisease/model/saved/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/PlantDisease/model/saved/best_model.keras
2281/2281 ━━━━━━━━━━━━━━━━━━━━ 1693s 719ms/step - accuracy: 0.8205 - loss: 0.6030 - val_accuracy: 0.9443 - val_loss: 0.1569 - learning_rate: 0.0010
Epoch 2/20
2280/2281 ━━━━━━━━━━━━━━━━━━━━ 0s 641ms/step - accuracy: 0.8790 - loss: 0.3822
Epoch 2: val_accuracy improved from 0.94429 to 0.95134, saving model to /content/drive/MyDrive/PlantDisease/model/saved/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/PlantDisease/model/saved/best_model.keras
2281/2281 ━━━━━━━━━━━━━━━━━━━━ 1529s 670ms/step - accuracy: 0.8846 - loss: 0.3692 - val_accuracy: 0.9513 - val_loss: 0.1423 - learning_rate: 0.0010
Epoch 3/20
228

## Step 7 — Phase 2: Fine-tune Top Backbone Layers

In [ ]:
print('=' * 50)
print('PHASE 2 — Fine-tuning top backbone layers')
print('=' * 50)

# Unfreeze top 30 layers of backbone
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False

# Recompile with very low learning rate
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f'Trainable parameters in Phase 2: {trainable:,}')

EPOCHS_PHASE2 = 15

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks
)

print('\nPhase 2 complete.')
print(f'Best val_accuracy : {max(history_phase2.history["val_accuracy"]):.4f}')